In [ ]:
import os,boto3,csv,fitz,csv,re,shutil

In [ ]:
txt_folder_name = r"C:\Users\abc\End_2_End_OCR_Through_S3_Bucket\GST_FULL_CODE\text_folder"
lst_txt = []

for folder in os.listdir(txt_folder_name):
    try:
        #print(folder)
        folder_path = os.path.join(txt_folder_name,folder)
        #print(folder_path)
        if os.path.isdir(folder_path):
            for filename in os.listdir(folder_path):
                if filename.split('.')[-1] in ('txt'):
                    if '.' in filename: 
                        full_path = os.path.join(folder_path,filename)
                        lst_txt.append(full_path)
                        
    except Exception as e:
        print(e)

len(lst_txt)

In [ ]:
pan_cid_info = []
i=0
for filepath in lst_txt:
    with open(filepath, 'r', newline='',encoding='UTF-8') as file:
        text = file.read()
        # pvt_pattern = re.compile(r"(\b.*Private Limited\b)|(\b.*limited\b)|(\b.*llp\b)|(\b.*Partnership\b)",re.IGNORECASE)
        if "Form GST REG" in text and not "INCOME TAX DEPARTMENT" in text.upper() :
            if not "Pay" in text.title():
                try:
                    gstin = re.findall(r"[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}[Z]{1}[0-9A-Z]{1}", text)
                    pan = list(set(re.findall(r"[a-zA-Z]{5}[0-9]{4}[A-Za-z]", text.upper())))
                    text2 = re.sub(r"[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}[Z]{1}[0-9A-Z]{1}","",text)
                    text3=re.sub(r"[\u0900-\u0D7F]+","",text2)
                    text4=re.sub(r"[a-zA-Z]{5}[0-9]{4}[A-Za-z]","\n",text3)
                    text5=re.sub(r"[0-9]{2}/[0-9]{2}/[0-9]{4}","\n",text4)
                    text6=re.sub(r"[0-9]+\n","",text5)
                    text7=re.sub(r"Date of Birth|\s?\/\s|\.|!|/|'\n|,|:|-|[(]|'\s|[0-9]+|(.+?)pan(.+?)+","",text6)
                    text8 = re.sub(r"\n\n*\s*","\n",text7).strip()
                    # name = list(set(pvt_pattern.findall(text8.title())))  
                    if ("PRIVATE" or "PVT" or "LTD") in text8.upper():
                        name=re.findall(r'(.+?) \n?PRIVATE ?(LIMITED)?',text8.upper())

                    elif "PRIVATE" not in text8:
                        if ("LIMITED" or "LTD") in text8:
                            name=re.findall(r'(.+?) ?\n?(LIMITED|LTD)',text8.upper())
                    elif "PRIVATE" not in text8:
                        if "LIMITED" not in text8:
                            if "PARTNERSHIP" in text8.upper():
                                name=re.findall(r'(.+?) ?\n?PARTNERSHIP',text8.upper())
                        
                    print(filepath)
                    print(name)
                    try:
                        if len(name[1]):
                            if len(name[0][0])<=len(name[1][0]):
                                namef=name[1]
                    except IndexError:
                        namef=name
                    
                    # print(pan)
                    # print(gstin)
                    # print(i,'----------------------------------------------------')
                    #print(i)
                    with open("GST_Format_text1.csv",'a',newline='',encoding='utf-8') as f:
                        writer =csv.writer(f)
                        writer.writerow([pan,gstin,namef,filepath])
                        #print("DONE")
                    
                    i+=1


                except Exception as e:
                    pan=None
                    gstin=None
                    name=None

                if gstin:
                    info={'gstin':gstin,'pan':pan,'name':name,'path':filepath}
                    pan_cid_info.append(info)    
print('CSV created Successfully')